## Objective
Inspect the raw source files before implementing the ETL pipeline.

- Identify all source files.
- Examine their structure and schemas.
- Measure missing values and duplicate records.
- Inspect identifiers, dates, and categorical values.
- Check consistency between timesheet records and the employee roster.
- Record findings for the Medallion Architecture design.

### Libraries


In [33]:
from pathlib import Path
import pandas as pd
import numpy as np
import csv
from IPython.display import display
import json


In [2]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 100)

### Paths

In [ ]:
PROJECT_DIR = Path.cwd().parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROFILE_DIR = PROJECT_DIR / "data" / "profiling"

#PROFILE_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Profling data directory:", PROFILE_DIR)

print("Raw directory exists:", RAW_DIR.exists())
print("Profiling directory exists:", PROFILE_DIR.exists())

Project directory: d:\ASE-ETL
Raw data directory: d:\ASE-ETL\data\raw
Profling data directory: d:\ASE-ETL\data\profiling
Raw directory exists: True
Profiling directory exists: True


### Listing sources


In [11]:
files = sorted(
    file for file in RAW_DIR.rglob("*")
    if file.is_file()
    and file.suffix.lower() in {".csv", ".txt", ".xlsx", ".xls"}
)

file_inventory = pd.DataFrame([
    {
        "filename": file.name,
        "extension": file.suffix.lower(),
        "size_kb": round(file.stat().st_size / 1024, 2),
        "relative_path": str(file.relative_to(RAW_DIR))
    }
    for file in files
])

display(file_inventory)

print(f"Total source files found: {len(files)}")

,filename,extension,size_kb,relative_path
0,employee_202510161125.csv,.csv,16.06,employee_202510161125.csv
1,timesheet_202509151540.csv,.csv,49760.26,timesheet_202509151540.csv
2,timesheet_202510161121.csv,.csv,7299.72,timesheet_202510161121.csv
3,timesheet_202510161122.csv,.csv,6606.31,timesheet_202510161122.csv
4,timesheet_202510161124.csv,.csv,3928.72,timesheet_202510161124.csv


Total source files found: 5


### Checking the delimiter, also parsing into columns


In [ ]:
def inspect_file(file_path, sample_size=8192):
    print("FILE:", file_path.name)

    if file_path.suffix.lower() in {".xlsx", ".xls"}:
        preview = pd.read_excel(file_path, nrows=5, dtype=str)
        print("Format: Excel")
        display(preview)
        return

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        errors="replace",
        newline=""
    ) as f:
        sample = f.read(sample_size)

    print("Raw preview:")
    print(sample[:1000])

    try:
        dialect = csv.Sniffer().sniff(
            sample,
            delimiters=",|\t;"
        )
        delimiter = dialect.delimiter
    except csv.Error:
        delimiter = ","

    print("Detected delimiter:", repr(delimiter))

    preview = pd.read_csv(
        file_path,
        sep=delimiter,
        nrows=5,
        dtype=str,
        encoding="utf-8-sig"
    )

    print("Parsed preview:")
    display(preview)


for file in files:
    inspect_file(file)

FILE: employee_202510161125.csv
Raw preview:
"client_employee_id"|"first_name"|"middle_name"|"last_name"|"preferred_name"|"job_code"|"job_title"|"job_start_date"|"organization_id"|"organization_name"|"department_id"|"department_name"|"dob"|"hire_date"|"recent_hire_date"|"anniversary_date"|"term_date"|"years_of_experience"|"work_email"|"address"|"city"|"state"|"zip"|"country"|"manager_employee_id"|"manager_employee_name"|"fte_status"|"is_per_deim"|"cell_phone"|"work_phone"|"scheduled_weekly_hour"|"active_status"|"termination_reason"|"clinical_level"
"00037"|Demo00037||Nurse|Demo00037|"1995"|Manager, CT Scan|"1998-06-01"|Lowell General Hospital|Lowell General Hospital|"72800"|"72800-CT Scan"|"1970-08-07"|"1998-06-01"|"1998-06-01"|"1998-06-01"|||nurse+00037@laudio.com|XXXXXXX|XXXXXXX|XXXXXXX||XXXXXXX|"09569"||Full_time||"978xxxxxxx"||"40"|"1"||
"507210"|Demo507210||Nurse|Demo507210|"1001"|Administrative Assistant II|"1994-01-11"|Care at Home|Care at Home|"214638"|"214638-Clinical Schedule

,client_employee_id,first_name,middle_name,last_name,preferred_name,job_code,job_title,job_start_date,organization_id,organization_name,department_id,department_name,dob,hire_date,recent_hire_date,anniversary_date,term_date,years_of_experience,work_email,address,city,state,zip,country,manager_employee_id,manager_employee_name,fte_status,is_per_deim,cell_phone,work_phone,scheduled_weekly_hour,active_status,termination_reason,clinical_level
0,00037,Demo00037,NaN,Nurse,Demo00037,1995,"Manager, CT Scan",1998-06-01,Lowell General Hospital,Lowell General Hospital,72800,72800-CT Scan,1970-08-07,1998-06-01,1998-06-01,1998-06-01,NaN,NaN,nurse+00037@laudio.com,XXXXXXX,XXXXXXX,XXXXXXX,NaN,XXXXXXX,09569,NaN,Full_time,NaN,978xxxxxxx,NaN,40,1,NaN,NaN
1,507210,Demo507210,NaN,Nurse,Demo507210,1001,Administrative Assistant II,1994-01-11,Care at Home,Care at Home,214638,214638-Clinical Scheduler Law Hospice,1963-03-17,1994-01-11,1994-01-11,1994-01-11,NaN,NaN,nurse+507210@laudio.com,XXXXXXX,XXXXXXX,XXXXXXX,NaN,XXXXXXX,507237,NaN,Full_time,NaN,978xxxxxxx,NaN,40,1,NaN,NaN
2,00080,Demo00080,NaN,Nurse,Demo00080,4377,Ultrasound Technologist III - PD,1998-07-16,Lowell General Hospital,Lowell General Hospital,72300,72300-Ultrasound,1960-02-14,1998-07-16,1998-07-16,1998-07-16,NaN,NaN,nurse+00080@laudio.com,XXXXXXX,XXXXXXX,XXXXXXX,NaN,XXXXXXX,05119,NaN,Variable_time,NaN,978xxxxxxx,NaN,40,1,NaN,NaN
3,503912,Demo503912,NaN,Nurse,Demo503912,2129,Nurse Assistant,1994-04-04,Lowell General Hospital,Lowell General Hospital,63200,63200-Medical Surgical Unit 2,1965-12-12,1994-04-04,1994-04-04,2024-10-29,2024-10-28,NaN,nurse+503912@laudio.com,XXXXXXX,XXXXXXX,XXXXXXX,NaN,XXXXXXX,03297,NaN,Full_time,NaN,978xxxxxxx,NaN,40,0,Terminate_Employee_Involuntary,NaN
4,00025,Demo00025,NaN,Nurse,Demo00025,1616,CT Technologist III,1998-06-01,Lowell General Hospital,Lowell General Hospital,72800,72800-CT Scan,1975-02-05,1998-06-01,1998-06-01,1998-06-01,NaN,NaN,nurse+00025@laudio.com,XXXXXXX,XXXXXXX,XXXXXXX,NaN,XXXXXXX,00037,NaN,Full_time,NaN,978xxxxxxx,NaN,40,1,NaN,NaN


FILE: timesheet_202509151540.csv
Raw preview:
"client_employee_id"|"department_id"|"department_name"|"home_department_id"|"home_department_name"|"pay_code"|"punch_in_comment"|"punch_out_comment"|"hours_worked"|"punch_apply_date"|"punch_in_datetime"|"punch_out_datetime"|"scheduled_start_datetime"|"scheduled_end_datetime"
"101005"|"1100-10013"|Nursing Labor and Delivery|"1100-10013"|Nursing Labor and Delivery|normal_worked|[NULL]|"EARLY"|8.50|"2025-05-28"|2025-05-28 10:53:00.000|2025-05-28 19:23:00.000||
"101031"|||"1100-10012"|Nursing Mother/Infant|HH-Meeting|[NULL]|[NULL]|1.00|"2025-05-08"|2025-05-08 00:00:00.000|2025-05-08 01:00:00.000||
"101069"|||"1100-13621"|MICU - MWH|HH-PTO sick|[NULL]|[NULL]|12.00|"2025-05-25"|2025-05-25 00:00:00.000|2025-05-25 12:00:00.000||
"101069"|||"1100-13621"|MICU - MWH|HH-PTO sick|[NULL]|[NULL]|12.00|"2025-06-30"|2025-06-30 00:00:00.000|2025-06-30 12:00:00.000||
"101073"|"1100-13621"|MICU - MWH|"1100-13621"|MICU - MWH|normal_worked|[NULL]|"LATE"|12.82|"2

,client_employee_id,department_id,department_name,home_department_id,home_department_name,pay_code,punch_in_comment,punch_out_comment,hours_worked,punch_apply_date,punch_in_datetime,punch_out_datetime,scheduled_start_datetime,scheduled_end_datetime
0,101005,1100-10013,Nursing Labor and Delivery,1100-10013,Nursing Labor and Delivery,normal_worked,[NULL],EARLY,8.50,2025-05-28,2025-05-28 10:53:00.000,2025-05-28 19:23:00.000,NaN,NaN
1,101031,NaN,NaN,1100-10012,Nursing Mother/Infant,HH-Meeting,[NULL],[NULL],1.00,2025-05-08,2025-05-08 00:00:00.000,2025-05-08 01:00:00.000,NaN,NaN
2,101069,NaN,NaN,1100-13621,MICU - MWH,HH-PTO sick,[NULL],[NULL],12.00,2025-05-25,2025-05-25 00:00:00.000,2025-05-25 12:00:00.000,NaN,NaN
3,101069,NaN,NaN,1100-13621,MICU - MWH,HH-PTO sick,[NULL],[NULL],12.00,2025-06-30,2025-06-30 00:00:00.000,2025-06-30 12:00:00.000,NaN,NaN
4,101073,1100-13621,MICU - MWH,1100-13621,MICU - MWH,normal_worked,[NULL],LATE,12.82,2025-06-22,2025-06-22 06:54:00.000,2025-06-22 19:43:00.000,2025-06-22 07:00:00.000,2025-06-22 19:30:00.000


FILE: timesheet_202510161121.csv
Raw preview:
"client_employee_id"|"department_id"|"department_name"|"home_department_id"|"home_department_name"|"pay_code"|"punch_in_comment"|"punch_out_comment"|"hours_worked"|"punch_apply_date"|"punch_in_datetime"|"punch_out_datetime"|"scheduled_start_datetime"|"scheduled_end_datetime"
"100715"||Nursing MaternalSpecial Care|||Hourly|[NULL]|"EARLY_OUT"|11.33|"2025-09-16"|2025-09-16 18:46:00.000|2025-09-17 06:06:00.000|2025-09-16 19:00:00.000|2025-09-17 07:30:00.000
"100715"||Nursing MaternalSpecial Care|||Hourly|"VERY_EARLY_IN"|"EARLY_OUT"|12.23|"2025-09-22"|2025-09-22 17:56:00.000|2025-09-23 06:10:00.000|2025-09-22 19:00:00.000|2025-09-23 07:30:00.000
"100785"|||||"DNU_Hourly Call Back|Hourly On Call"|[NULL]|"EARLY_OUT|SHORT_SHIFT"|10.20|"2025-09-19"|2025-09-19 20:48:00.000|2025-09-20 07:00:00.000|2025-09-19 21:00:00.000|2025-09-20 07:00:00.000
"100715"||Nursing MaternalSpecial Care|||Hourly|"UNSCHEDULED"|[NULL]|12.43|"2025-09-18"|2025-09-18 17:43:00.

,client_employee_id,department_id,department_name,home_department_id,home_department_name,pay_code,punch_in_comment,punch_out_comment,hours_worked,punch_apply_date,punch_in_datetime,punch_out_datetime,scheduled_start_datetime,scheduled_end_datetime
0,100715,NaN,Nursing MaternalSpecial Care,NaN,NaN,Hourly,[NULL],EARLY_OUT,11.33,2025-09-16,2025-09-16 18:46:00.000,2025-09-17 06:06:00.000,2025-09-16 19:00:00.000,2025-09-17 07:30:00.000
1,100715,NaN,Nursing MaternalSpecial Care,NaN,NaN,Hourly,VERY_EARLY_IN,EARLY_OUT,12.23,2025-09-22,2025-09-22 17:56:00.000,2025-09-23 06:10:00.000,2025-09-22 19:00:00.000,2025-09-23 07:30:00.000
2,100785,NaN,NaN,NaN,NaN,DNU_Hourly Call Back|Hourly On Call,[NULL],EARLY_OUT|SHORT_SHIFT,10.20,2025-09-19,2025-09-19 20:48:00.000,2025-09-20 07:00:00.000,2025-09-19 21:00:00.000,2025-09-20 07:00:00.000
3,100715,NaN,Nursing MaternalSpecial Care,NaN,NaN,Hourly,UNSCHEDULED,[NULL],12.43,2025-09-18,2025-09-18 17:43:00.000,2025-09-19 06:09:00.000,NaN,NaN
4,100785,NaN,OP Recovery -PACU,NaN,NaN,Hourly,[NULL],[NULL],10.52,2025-09-18,2025-09-18 09:03:00.000,2025-09-18 19:34:00.000,2025-09-18 09:00:00.000,2025-09-18 19:30:00.000


FILE: timesheet_202510161122.csv
Raw preview:
"client_employee_id"|"department_id"|"department_name"|"home_department_id"|"home_department_name"|"pay_code"|"punch_in_comment"|"punch_out_comment"|"hours_worked"|"punch_apply_date"|"punch_in_datetime"|"punch_out_datetime"|"scheduled_start_datetime"|"scheduled_end_datetime"
"100715"||Nursing MaternalSpecial Care|||Hourly|"VERY_EARLY_IN"|"EARLY_OUT"|12.23|"2025-09-22"|2025-09-22 17:56:00.000|2025-09-23 06:10:00.000|2025-09-22 19:00:00.000|2025-09-23 07:30:00.000
"100801"|||||DNU_Hourly Call Back|[NULL]|[NULL]|24.00|"2025-09-27"|2025-09-27 07:00:00.000|2025-09-28 07:00:00.000||
"100801"|||||DNU_Hourly Call Back|[NULL]|[NULL]|24.00|"2025-09-28"|2025-09-28 07:00:00.000|2025-09-29 07:00:00.000||
"100801"|||||Hourly|[NULL]|[NULL]|8.50|"2025-10-03"|2025-10-03 07:30:00.000|2025-10-03 16:00:00.000||
"100801"||Adult Endoscopy|||Hourly|[NULL]|[NULL]|8.57|"2025-09-23"|2025-09-23 06:56:00.000|2025-09-23 15:30:00.000|2025-09-23 07:00:00.000|2025-09-23 1

,client_employee_id,department_id,department_name,home_department_id,home_department_name,pay_code,punch_in_comment,punch_out_comment,hours_worked,punch_apply_date,punch_in_datetime,punch_out_datetime,scheduled_start_datetime,scheduled_end_datetime
0,100715,NaN,Nursing MaternalSpecial Care,NaN,NaN,Hourly,VERY_EARLY_IN,EARLY_OUT,12.23,2025-09-22,2025-09-22 17:56:00.000,2025-09-23 06:10:00.000,2025-09-22 19:00:00.000,2025-09-23 07:30:00.000
1,100801,NaN,NaN,NaN,NaN,DNU_Hourly Call Back,[NULL],[NULL],24.00,2025-09-27,2025-09-27 07:00:00.000,2025-09-28 07:00:00.000,NaN,NaN
2,100801,NaN,NaN,NaN,NaN,DNU_Hourly Call Back,[NULL],[NULL],24.00,2025-09-28,2025-09-28 07:00:00.000,2025-09-29 07:00:00.000,NaN,NaN
3,100801,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],8.50,2025-10-03,2025-10-03 07:30:00.000,2025-10-03 16:00:00.000,NaN,NaN
4,100801,NaN,Adult Endoscopy,NaN,NaN,Hourly,[NULL],[NULL],8.57,2025-09-23,2025-09-23 06:56:00.000,2025-09-23 15:30:00.000,2025-09-23 07:00:00.000,2025-09-23 15:30:00.000


FILE: timesheet_202510161124.csv
Raw preview:
"client_employee_id"|"department_id"|"department_name"|"home_department_id"|"home_department_name"|"pay_code"|"punch_in_comment"|"punch_out_comment"|"hours_worked"|"punch_apply_date"|"punch_in_datetime"|"punch_out_datetime"|"scheduled_start_datetime"|"scheduled_end_datetime"
"100715"|||||Hourly|[NULL]|[NULL]|12.50|"2025-10-14"|2025-10-14 19:00:00.000|2025-10-15 07:30:00.000||
"100801"|||||Hourly|[NULL]|[NULL]|8.50|"2025-10-09"|2025-10-09 07:30:00.000|2025-10-09 16:00:00.000||
"100801"|||||Hourly|[NULL]|[NULL]|8.50|"2025-10-10"|2025-10-10 07:00:00.000|2025-10-10 15:30:00.000||
"100715"|||||Hourly|[NULL]|[NULL]|12.50|"2025-10-04"|2025-10-04 07:00:00.000|2025-10-04 19:30:00.000||
"100715"|||||Hourly|[NULL]|[NULL]|12.50|"2025-10-10"|2025-10-10 07:00:00.000|2025-10-10 19:30:00.000||
"100715"|||||Hourly|[NULL]|[NULL]|12.50|"2025-10-11"|2025-10-11 19:00:00.000|2025-10-12 07:30:00.000||
"100715"|||||Hourly|[NULL]|[NULL]|6.00|"2025-10-07"|2025-10-07

,client_employee_id,department_id,department_name,home_department_id,home_department_name,pay_code,punch_in_comment,punch_out_comment,hours_worked,punch_apply_date,punch_in_datetime,punch_out_datetime,scheduled_start_datetime,scheduled_end_datetime
0,100715,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],12.50,2025-10-14,2025-10-14 19:00:00.000,2025-10-15 07:30:00.000,NaN,NaN
1,100801,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],8.50,2025-10-09,2025-10-09 07:30:00.000,2025-10-09 16:00:00.000,NaN,NaN
2,100801,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],8.50,2025-10-10,2025-10-10 07:00:00.000,2025-10-10 15:30:00.000,NaN,NaN
3,100715,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],12.50,2025-10-04,2025-10-04 07:00:00.000,2025-10-04 19:30:00.000,NaN,NaN
4,100715,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],12.50,2025-10-10,2025-10-10 07:00:00.000,2025-10-10 19:30:00.000,NaN,NaN


### Loading for profiling(load as strings)

In [ ]:
# loading as strings in order to inspect he og vales before deciding how to convert the dates,fields,etc
def load_source(file_path):
    if file_path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(file_path, dtype=str)

    with open(
        file_path,
        "r",
        encoding="utf-8-sig",
        errors="replace",
        newline=""
    ) as f:
        sample = f.read(8192)

    try:
        delimiter = csv.Sniffer().sniff(
            sample,
            delimiters=",|\t;"
        ).delimiter
    except csv.Error:
        delimiter = ","

    return pd.read_csv(
        file_path,
        sep=delimiter,
        dtype=str,
        encoding="utf-8-sig",
        low_memory=False
    )


dataframes = {}

for file in files:
    df = load_source(file)
    dataframes[file.name] = df

    print(
        f"{file.name}: "
        f"{df.shape[0]:,} rows × {df.shape[1]} columns"
    )

employee_202510161125.csv: 50 rows × 34 columns
timesheet_202509151540.csv: 269,921 rows × 14 columns
timesheet_202510161121.csv: 48,258 rows × 14 columns
timesheet_202510161122.csv: 55,435 rows × 14 columns
timesheet_202510161124.csv: 38,957 rows × 14 columns


In [ ]:
print(list(dataframes.keys()))
# the dataframes dictionary now contains 1 df per source file

['employee_202510161125.csv', 'timesheet_202509151540.csv', 'timesheet_202510161121.csv', 'timesheet_202510161122.csv', 'timesheet_202510161124.csv']


### Compare row and column counts

In [16]:
structure = pd.DataFrame([
    {
        "filename": filename,
        "rows": len(df),
        "columns": len(df.columns),
        "memory_mb": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2), 2
        )
    }
    for filename, df in dataframes.items()
])

display(structure)

,filename,rows,columns,memory_mb
0,employee_202510161125.csv,50,34,0.08
1,timesheet_202509151540.csv,269921,14,213.14
2,timesheet_202510161121.csv,48258,14,34.98
3,timesheet_202510161122.csv,55435,14,37.13
4,timesheet_202510161124.csv,38957,14,24.86


### Inspecting the columns

In [18]:
for filename, df in dataframes.items():
    print(f"\n{'=' * 80}")
    print(filename)

    schema = pd.DataFrame({
        "column": df.columns,
        "pandas_dtype": [str(df[col].dtype) for col in df.columns],
        "non_null_count": [df[col].notna().sum() for col in df.columns],
        "distinct_values": [df[col].nunique(dropna=True) for col in df.columns]
    })

    display(schema)

# data type will be reported as strings as because we loaded the files as string 


employee_202510161125.csv


,column,pandas_dtype,non_null_count,distinct_values
0,client_employee_id,str,50,50
1,first_name,str,50,50
2,middle_name,str,0,0
3,last_name,str,50,1
4,preferred_name,str,50,50
5,job_code,str,50,36
6,job_title,str,50,36
7,job_start_date,str,50,48
8,organization_id,str,50,3
9,organization_name,str,50,3



timesheet_202509151540.csv


,column,pandas_dtype,non_null_count,distinct_values
0,client_employee_id,str,269921,9792
1,department_id,str,222006,621
2,department_name,str,221933,500
3,home_department_id,str,262593,557
4,home_department_name,str,262593,497
5,pay_code,str,269921,669
6,punch_in_comment,str,269921,1395
7,punch_out_comment,str,269921,90
8,hours_worked,str,269921,1177
9,punch_apply_date,str,269921,137



timesheet_202510161121.csv


,column,pandas_dtype,non_null_count,distinct_values
0,client_employee_id,str,48258,8303
1,department_id,str,24388,374
2,department_name,str,24830,411
3,home_department_id,str,0,0
4,home_department_name,str,0,0
5,pay_code,str,48258,141
6,punch_in_comment,str,48258,411
7,punch_out_comment,str,48258,96
8,hours_worked,str,48258,1039
9,punch_apply_date,str,48258,11



timesheet_202510161122.csv


,column,pandas_dtype,non_null_count,distinct_values
0,client_employee_id,str,55435,8270
1,department_id,str,8994,363
2,department_name,str,9395,391
3,home_department_id,str,0,0
4,home_department_name,str,0,0
5,pay_code,str,55435,111
6,punch_in_comment,str,55435,108
7,punch_out_comment,str,55435,54
8,hours_worked,str,55435,872
9,punch_apply_date,str,55435,15



timesheet_202510161124.csv


,column,pandas_dtype,non_null_count,distinct_values
0,client_employee_id,str,38957,6411
1,department_id,str,0,0
2,department_name,str,1,1
3,home_department_id,str,0,0
4,home_department_name,str,0,0
5,pay_code,str,38957,61
6,punch_in_comment,str,38957,1
7,punch_out_comment,str,38957,2
8,hours_worked,str,38957,80
9,punch_apply_date,str,38957,13


### Checking for the missing values

In [19]:
for filename, df in dataframes.items():
    print(f"\nMissing values: {filename}")

    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": (
            df.isna().mean() * 100
        ).round(2)
    }).sort_values("missing_percentage", ascending=False)

    display(missing[missing["missing_count"] > 0])


Missing values: employee_202510161125.csv


,missing_count,missing_percentage
years_of_experience,50,100.0
middle_name,50,100.0
is_per_deim,50,100.0
work_phone,50,100.0
zip,50,100.0
clinical_level,50,100.0
manager_employee_name,50,100.0
term_date,44,88.0
termination_reason,44,88.0



Missing values: timesheet_202509151540.csv


,missing_count,missing_percentage
scheduled_end_datetime,138409,51.28
scheduled_start_datetime,138409,51.28
department_name,47988,17.78
department_id,47915,17.75
home_department_name,7328,2.71
home_department_id,7328,2.71



Missing values: timesheet_202510161121.csv


,missing_count,missing_percentage
home_department_name,48258,100.00
home_department_id,48258,100.00
department_id,23870,49.46
department_name,23428,48.55
scheduled_end_datetime,14695,30.45
scheduled_start_datetime,14695,30.45



Missing values: timesheet_202510161122.csv


,missing_count,missing_percentage
home_department_name,55435,100.00
home_department_id,55435,100.00
department_id,46441,83.78
department_name,46040,83.05
scheduled_end_datetime,40394,72.87
scheduled_start_datetime,40394,72.87



Missing values: timesheet_202510161124.csv


,missing_count,missing_percentage
department_id,38957,100.0
department_name,38956,100.0
home_department_name,38957,100.0
home_department_id,38957,100.0
scheduled_end_datetime,38957,100.0
scheduled_start_datetime,38957,100.0


#### For values like XXXXXXX or 986xxxxxxx

In [27]:
missing_markers = {
    "NULL", "N/A", "NA", "NAN",
    "NONE", "UNKNOWN", "NOT AVAILABLE"
}

for filename, df in dataframes.items():
    print(f"\n{'=' * 80}")
    print(f"File: {filename}")

    # Normalize text column by column
    normalized = df.astype("string").apply(
        lambda col: col.str.strip().str.upper()
    )

    # Actual missing values
    actual_missing = df.isna()

    # Empty strings
    empty_strings = normalized.eq("").fillna(False)

    # Common textual missing markers
    text_markers = normalized.isin(missing_markers)

    # Initialize masks with the same shape as the DataFrame
    all_x = pd.DataFrame(
        False, index=df.index, columns=df.columns
    )
    contains_x = pd.DataFrame(
        False, index=df.index, columns=df.columns
    )

    # Apply string operations to each column (Series)
    for column in normalized.columns:
        col = normalized[column]

        all_x[column] = col.str.fullmatch(
            r"X+", na=False
        )

        contains_x[column] = col.str.contains(
            r"X", na=False, regex=True
        ) & ~all_x[column]

    # Summarize findings
    report = pd.DataFrame({
        "actual_missing": actual_missing.sum(),
        "empty_strings": empty_strings.sum(),
        "text_markers": text_markers.sum(),
        "all_x_values": all_x.sum(),
        "values_containing_x": contains_x.sum()
    })

    display(
        report[report.sum(axis=1) > 0]
        .sort_values(
            "values_containing_x",
            ascending=False
        )
    )

    # Display examples for investigation
    for column in df.columns:
        mask = all_x[column] | contains_x[column]

        if mask.any():
            print(f"\nColumn: {column}")
            print("Examples of values containing X:")

            display(
                df.loc[mask, [column]]
                .drop_duplicates()
                .head(20)
            )



File: employee_202510161125.csv


,actual_missing,empty_strings,text_markers,all_x_values,values_containing_x
cell_phone,0,0,0,0,50
job_title,0,0,0,0,1
middle_name,50,0,0,0,0
years_of_experience,50,0,0,0,0
address,0,0,0,50,0
city,0,0,0,50,0
term_date,44,0,0,0,0
state,0,0,0,50,0
zip,50,0,0,0,0
manager_employee_name,50,0,0,0,0



Column: job_title
Examples of values containing X:


,job_title
24,RN IV Complex Care



Column: address
Examples of values containing X:


...

,punch_out_comment
25514,30 Minutes after 6.25|CANCELLED_DEDUCTION|Extra Shift - Note Required
39173,Extra Shift - Note Required



File: timesheet_202510161124.csv


,actual_missing,empty_strings,text_markers,all_x_values,values_containing_x
pay_code,0,0,0,0,2702
department_name,38956,0,0,0,0
department_id,38957,0,0,0,0
home_department_id,38957,0,0,0,0
home_department_name,38957,0,0,0,0
scheduled_start_datetime,38957,0,0,0,0
scheduled_end_datetime,38957,0,0,0,0



Column: pay_code
Examples of values containing X:


,pay_code
240,Exempt


### Checking duplicates

In [22]:
for filename, df in dataframes.items():
    exact_duplicates = int(df.duplicated().sum())

    print(f"\nFile: {filename}")
    print(f"Exact duplicate rows: {exact_duplicates:,}")

    if exact_duplicates > 0:
        display(df[df.duplicated(keep=False)].head(10))


File: employee_202510161125.csv
Exact duplicate rows: 0

File: timesheet_202509151540.csv
Exact duplicate rows: 0

File: timesheet_202510161121.csv
Exact duplicate rows: 0

File: timesheet_202510161122.csv
Exact duplicate rows: 0

File: timesheet_202510161124.csv
Exact duplicate rows: 0


### Cross-file overlapp analysis

In [28]:
combined = pd.concat(
    [
        df.assign(source_file=filename)
        for filename, df in dataframes.items()
    ],
    ignore_index=True
)

print("Total combined rows:", len(combined))
print("Total source rows:", sum(len(df) for df in dataframes.values()))

Total combined rows: 412621
Total source rows: 412621


In [29]:
data_columns = [
    col for col in combined.columns
    if col != "source_file"
]

cross_file_exact = combined[
    combined.duplicated(
        subset=data_columns,
        keep=False
    )
].sort_values(data_columns)

print(
    "Rows involved in exact cross-file duplicates:",
    len(cross_file_exact)
)

display(cross_file_exact.head(30))

Rows involved in exact cross-file duplicates: 61678


,client_employee_id,first_name,middle_name,last_name,preferred_name,job_code,job_title,job_start_date,organization_id,organization_name,department_id,department_name,dob,hire_date,recent_hire_date,anniversary_date,term_date,years_of_experience,work_email,address,city,state,zip,country,manager_employee_id,manager_employee_name,fte_status,is_per_deim,cell_phone,work_phone,scheduled_weekly_hour,active_status,termination_reason,clinical_level,source_file,home_department_id,home_department_name,pay_code,punch_in_comment,punch_out_comment,hours_worked,punch_apply_date,punch_in_datetime,punch_out_datetime,scheduled_start_datetime,scheduled_end_datetime
277901,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,timesheet_202510161121.csv,NaN,NaN,Hourly,LATE_IN,LATE_OUT,8.53,2025-09-22,2025-09-22 07:30:00.000,2025-09-22 16:02:00.000,2025-09-22 07:00:00.000,2025-09-22 15:30:00.000
327421,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,timesheet_202510161122.csv,NaN,NaN,Hourly,LATE_IN,LATE_OUT,8.53,2025-09-22,2025-09-22 07:30:00.000,2025-09-22 16:02:00.000,2025-09-22 07:00:00.000,2025-09-22 15:30:00.000
277899,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,timesheet_202510161121.csv,NaN,NaN,Hourly,[NULL],EARLY_OUT,7.52,2025-09-24,2025-09-24 07:05:00.000,2025-09-24 14:36:00.000,2025-09-24 07:00:00.000,2025-09-24 15:30:00.000
327420,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,timesheet_202510161122.csv,NaN,NaN,Hourly,[NULL],EARLY_OUT,7.52,2025-09-24,2025-09-24 07:05:00.000,2025-09-24 14:36:00.000,2025-09-24 07:00:00.000,2025-09-24 15:30:00.000
277898,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,timesheet_202510161121.csv,NaN,NaN,Hourly,[NULL],[NULL],8.52,2025-09-23,2025-09-23 07:02:00.000,2025-09-23 15:33:00.000,2025-09-23 07:00:00.000,2025-09-23 15:30:00.000
327419,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,timesheet_202510161122.csv,NaN,NaN,Hourly,[NULL],[NULL],8.52,2025-09-23,2025-09-23 07:02:00.000,2025-09-23 15:33:00.000,2025-09-23 07:00:00.000,2025-09-23 15:30:00.000
301100,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,timesheet_202510161121.csv,NaN,NaN,Hourly,[NULL],[NULL],8.50,2025-09-25,2025-09-25 07:00:00.000,2025-09-25 15:30:00.000,2025-09-25 07:00:00.000,2025-09-25 15:30:00.000
354009,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,timesheet_202510161122.csv,NaN,NaN,Hourly,[NULL],[NULL],8.50,2025-09-25,2025-09-25 07:00:00.000,2025-09-25 15:30:00.000,2025-09-25 07:00:00.000,2025-09-25 15:30:00.000
354012,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,timesheet_202510161122.csv,NaN,NaN,Hourly,[NULL],[NULL],8.50,2025-10-06,2025-10-06 07:00:00.000,2025-10-06 15:30:00.000,NaN,NaN
398933,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,timesheet_202510161124.csv,NaN,NaN,Hourly,[NULL],[NULL],8.50,2025-10-06,2025-10-06 07:00:00.000,2025-10-06 15:30:00.000,NaN,NaN


In [30]:
duplicate_groups = (
    cross_file_exact
    .groupby(data_columns, dropna=False)["source_file"]
    .agg(list)
    .reset_index(name="source_files")
)

duplicate_groups = duplicate_groups[
    duplicate_groups["source_files"].apply(
        lambda sources: len(set(sources)) > 1
    )
]

display(duplicate_groups.head(30))

,client_employee_id,first_name,middle_name,last_name,preferred_name,job_code,job_title,job_start_date,organization_id,organization_name,department_id,department_name,dob,hire_date,recent_hire_date,anniversary_date,term_date,years_of_experience,work_email,address,city,state,zip,country,manager_employee_id,manager_employee_name,fte_status,is_per_deim,cell_phone,work_phone,scheduled_weekly_hour,active_status,termination_reason,clinical_level,home_department_id,home_department_name,pay_code,punch_in_comment,punch_out_comment,hours_worked,punch_apply_date,punch_in_datetime,punch_out_datetime,scheduled_start_datetime,scheduled_end_datetime,source_files
0,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hourly,LATE_IN,LATE_OUT,8.53,2025-09-22,2025-09-22 07:30:00.000,2025-09-22 16:02:00.000,2025-09-22 07:00:00.000,2025-09-22 15:30:00.000,"[timesheet_202510161121.csv, timesheet_202510161122.csv]"
1,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hourly,[NULL],EARLY_OUT,7.52,2025-09-24,2025-09-24 07:05:00.000,2025-09-24 14:36:00.000,2025-09-24 07:00:00.000,2025-09-24 15:30:00.000,"[timesheet_202510161121.csv, timesheet_202510161122.csv]"
2,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,72800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],8.52,2025-09-23,2025-09-23 07:02:00.000,2025-09-23 15:33:00.000,2025-09-23 07:00:00.000,2025-09-23 15:30:00.000,"[timesheet_202510161121.csv, timesheet_202510161122.csv]"
3,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],8.50,2025-09-25,2025-09-25 07:00:00.000,2025-09-25 15:30:00.000,2025-09-25 07:00:00.000,2025-09-25 15:30:00.000,"[timesheet_202510161121.csv, timesheet_202510161122.csv]"
4,00025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],8.50,2025-10-06,2025-10-06 07:00:00.000,2025-10-06 15:30:00.000,NaN,NaN,"[timesheet_202510161122.csv, timesheet_202510161124.csv]"
5,00068,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],7.00,2025-09-26,2025-09-26 16:00:00.000,2025-09-26 23:00:00.000,NaN,NaN,"[timesheet_202510161121.csv, timesheet_202510161122.csv]"
6,00068,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],7.00,2025-10-04,2025-10-04 16:00:00.000,2025-10-04 23:00:00.000,NaN,NaN,"[timesheet_202510161122.csv, timesheet_202510161124.csv]"
7,00068,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],7.00,2025-10-05,2025-10-05 16:00:00.000,2025-10-05 23:00:00.000,NaN,NaN,"[timesheet_202510161122.csv, timesheet_202510161124.csv]"
8,00131,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],8.60,2025-09-22,2025-09-22 08:24:00.000,2025-09-22 17:00:00.000,2025-09-22 08:30:00.000,2025-09-22 17:00:00.000,"[timesheet_202510161121.csv, timesheet_202510161122.csv]"
9,00131,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hourly,[NULL],[NULL],8.50,2025-09-25,2025-09-25 08:30:00.000,2025-09-25 17:00:00.000,2025-09-25 08:30:00.000,2025-09-25 17:00:00.000,"[timesheet_202510161121.csv, timesheet_202510161122.csv]"


### Inspecting identifiers and categorical cols

In [24]:
for filename, df in dataframes.items():
    print(f"\nCategorical-value investigation: {filename}")

    for column in df.columns:
        unique_count = df[column].nunique(dropna=True)

        if 1 < unique_count <= 30:
            print(f"\n{column} ({unique_count} distinct values)")
            display(
                df[column]
                .value_counts(dropna=False)
                .head(30)
                .rename_axis(column)
                .reset_index(name="count")
            )


Categorical-value investigation: employee_202510161125.csv

organization_id (3 distinct values)


,organization_id,count
0,Care at Home,30
1,Lowell General Hospital,17
2,Tufts Medicine,3



organization_name (3 distinct values)


,organization_name,count
0,Care at Home,30
1,Lowell General Hospital,17
2,Tufts Medicine,3



term_date (6 distinct values)


...

,punch_apply_date,count
0,2025-09-23,5052
1,2025-09-22,4976
2,2025-09-25,4775
3,2025-09-24,4648
4,2025-09-26,4441
5,2025-10-02,4339
6,2025-09-30,4327
7,2025-10-01,4312
8,2025-09-29,4270
9,2025-10-03,4071



Categorical-value investigation: timesheet_202510161124.csv

punch_out_comment (2 distinct values)


,punch_out_comment,count
0,[NULL],38956
1,VERY_LATE_OUT,1



punch_apply_date (13 distinct values)


,punch_apply_date,count
0,2025-10-09,4042
1,2025-10-08,4005
2,2025-10-07,4001
3,2025-10-06,3971
4,2025-10-10,3820
5,2025-10-16,3428
6,2025-10-14,3412
7,2025-10-15,3402
8,2025-10-13,3319
9,2025-10-04,1537


In [25]:
for filename, df in dataframes.items():
    print(f"\nIdentifier investigation: {filename}")

    for column in df.columns:
        name = column.lower()

        if any(term in name for term in ["employee", "worker", "staff", "id"]):
            print(
                f"{column}: "
                f"{df[column].nunique(dropna=True):,} unique non-null values"
            )
            display(df[column].dropna().head(10))


Identifier investigation: employee_202510161125.csv
client_employee_id: 50 unique non-null values


0     00037
1    507210
2     00080
3    503912
4     00025
5    506323
6     00068
7    505689
8     00131
9     00214
Name: client_employee_id, dtype: str

middle_name: 0 unique non-null values


Series([], Name: middle_name, dtype: str)

organization_id: 3 unique non-null values


...

0    100715
1    100801
2    100801
3    100715
4    100715
5    100715
6    100715
7    100715
8    100715
9    101005
Name: client_employee_id, dtype: str

department_id: 0 unique non-null values


Series([], Name: department_id, dtype: str)

home_department_id: 0 unique non-null values


Series([], Name: home_department_id, dtype: str)

### Investigate date and numeric candidates

In [26]:
for filename, df in dataframes.items():
    print(f"\nDate and numeric candidates: {filename}")

    for column in df.columns:
        name = column.lower()

        if any(term in name for term in ["date", "time", "start", "end"]):
            parsed = pd.to_datetime(
                df[column],
                errors="coerce"
            )

            non_missing = df[column].notna().sum()
            valid = parsed.notna().sum()

            print(
                f"{column}: "
                f"{valid:,}/{non_missing:,} non-null values parsed as dates/times"
            )

        if any(term in name for term in ["hour", "amount", "rate", "pay", "wage"]):
            numeric = pd.to_numeric(
                df[column],
                errors="coerce"
            )

            print(f"\nNumeric candidate: {column}")
            print("Non-null source values:", int(df[column].notna().sum()))
            print("Successfully parsed numeric:", int(numeric.notna().sum()))
            print("Minimum:", numeric.min())
            print("Maximum:", numeric.max())


Date and numeric candidates: employee_202510161125.csv
job_start_date: 50/50 non-null values parsed as dates/times
hire_date: 50/50 non-null values parsed as dates/times
recent_hire_date: 50/50 non-null values parsed as dates/times
anniversary_date: 50/50 non-null values parsed as dates/times
term_date: 6/6 non-null values parsed as dates/times

Numeric candidate: scheduled_weekly_hour
Non-null source values: 50
Successfully parsed numeric: 50
Minimum: 40
Maximum: 40

Date and numeric candidates: timesheet_202509151540.csv

Numeric candidate: pay_code
Non-null source values: 269921
Successfully parsed numeric: 0
Minimum: nan
Maximum: nan

Numeric candidate: hours_worked
Non-null source values: 269921
Successfully parsed numeric: 269921
Minimum: 0.02
Maximum: 26.0
punch_apply_date: 269,921/269,921 non-null values parsed as dates/times
punch_in_datetime: 269,921/269,921 non-null values parsed as dates/times
punch_out_datetime: 269,921/269,921 non-null values parsed as dates/times
schedu

### Save initial inspection results

In [31]:
summary_rows = []

for filename, df in dataframes.items():
    for column in df.columns:
        summary_rows.append({
            "filename": filename,
            "column": column,
            "row_count": len(df),
            "missing_count": int(df[column].isna().sum()),
            "missing_percentage": round(
                df[column].isna().mean() * 100, 2
            ),
            "distinct_values": int(df[column].nunique(dropna=True))
        })

summary = pd.DataFrame(summary_rows)

output_path = PROFILE_DIR / "initial_column_profile.csv"
summary.to_csv(output_path, index=False)

print("Saved:", output_path.resolve())
display(summary)

Saved: D:\ASE-ETL\data\profiling\initial_column_profile.csv


,filename,column,row_count,missing_count,missing_percentage,distinct_values
0,employee_202510161125.csv,client_employee_id,50,0,0.0,50
1,employee_202510161125.csv,first_name,50,0,0.0,50
2,employee_202510161125.csv,middle_name,50,50,100.0,0
3,employee_202510161125.csv,last_name,50,0,0.0,1
4,employee_202510161125.csv,preferred_name,50,0,0.0,50
...,...,...,...,...,...,...
85,timesheet_202510161124.csv,punch_apply_date,38957,0,0.0,13
86,timesheet_202510161124.csv,punch_in_datetime,38957,0,0.0,711
87,timesheet_202510161124.csv,punch_out_datetime,38957,0,0.0,749
88,timesheet_202510161124.csv,scheduled_start_datetime,38957,38957,100.0,0


In [34]:
for filename, df in dataframes.items():
    report = {
        "filename": filename,
        "row_count": int(len(df)),
        "column_count": int(len(df.columns)),
        "columns": [
            {
                "name": col,
                "loaded_dtype": str(df[col].dtype),
                "null_count": int(df[col].isna().sum()),
                "distinct_count": int(df[col].nunique(dropna=True)),
                "exact_duplicate_rows_involving_value": None
            }
            for col in df.columns
        ],
        "exact_duplicate_rows": int(df.duplicated().sum())
    }

    output_file = PROFILE_DIR / f"{Path(filename).stem}_profile.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=4, ensure_ascii=False)

    print("Saved:", output_file.name)

Saved: employee_202510161125_profile.json
Saved: timesheet_202509151540_profile.json
Saved: timesheet_202510161121_profile.json
Saved: timesheet_202510161122_profile.json
Saved: timesheet_202510161124_profile.json


### Primary and foreign key candidate analysis

In [35]:
key_candidates = []

for filename, df in dataframes.items():
    for column in df.columns:
        values = df[column].astype("string").str.strip()

        non_missing = values.notna() & values.ne("")
        valid_values = values[non_missing]

        key_candidates.append({
            "filename": filename,
            "column": column,
            "row_count": len(df),
            "missing_count": int((~non_missing).sum()),
            "distinct_non_missing": int(valid_values.nunique()),
            "duplicate_non_missing_values": int(
                valid_values.duplicated().sum()
            ),
            "unique_percentage": round(
                valid_values.nunique() / max(len(valid_values), 1) * 100,
                2
            )
        })

key_report = pd.DataFrame(key_candidates)

display(
    key_report.sort_values(
        ["filename", "unique_percentage"],
        ascending=[True, False]
    )
)

key_report.to_csv(
    PROFILE_DIR / "key_candidate_profile.csv",
    index=False
)

,filename,column,row_count,missing_count,distinct_non_missing,duplicate_non_missing_values,unique_percentage
0,employee_202510161125.csv,client_employee_id,50,0,50,0,100.0
1,employee_202510161125.csv,first_name,50,0,50,0,100.0
4,employee_202510161125.csv,preferred_name,50,0,50,0,100.0
12,employee_202510161125.csv,dob,50,0,50,0,100.0
16,employee_202510161125.csv,term_date,50,44,6,0,100.0
...,...,...,...,...,...,...,...
79,timesheet_202510161124.csv,home_department_id,38957,38957,0,0,0.0
80,timesheet_202510161124.csv,home_department_name,38957,38957,0,0,0.0
82,timesheet_202510161124.csv,punch_in_comment,38957,0,1,38956,0.0
88,timesheet_202510161124.csv,scheduled_start_datetime,38957,38957,0,0,0.0


In [39]:
EMPLOYEE_FILE = "employee_202510161125.csv"
EMPLOYEE_ID_COLUMN = "client_employee_id"

employee_df = dataframes[EMPLOYEE_FILE]

roster_ids = set(
    employee_df[EMPLOYEE_ID_COLUMN]
    .dropna()
    .astype("string")
    .str.strip()
)

roster_ids.discard("")

In [41]:
for filename, df in dataframes.items():
    print(f"\nFile: {filename}")
    print("Columns:", df.columns.tolist())


File: employee_202510161125.csv
Columns: ['client_employee_id', 'first_name', 'middle_name', 'last_name', 'preferred_name', 'job_code', 'job_title', 'job_start_date', 'organization_id', 'organization_name', 'department_id', 'department_name', 'dob', 'hire_date', 'recent_hire_date', 'anniversary_date', 'term_date', 'years_of_experience', 'work_email', 'address', 'city', 'state', 'zip', 'country', 'manager_employee_id', 'manager_employee_name', 'fte_status', 'is_per_deim', 'cell_phone', 'work_phone', 'scheduled_weekly_hour', 'active_status', 'termination_reason', 'clinical_level']

File: timesheet_202509151540.csv
Columns: ['client_employee_id', 'department_id', 'department_name', 'home_department_id', 'home_department_name', 'pay_code', 'punch_in_comment', 'punch_out_comment', 'hours_worked', 'punch_apply_date', 'punch_in_datetime', 'punch_out_datetime', 'scheduled_start_datetime', 'scheduled_end_datetime']

File: timesheet_202510161121.csv
Columns: ['client_employee_id', 'department_i

In [42]:
TIMESHEET_ID_COLUMNS = {
    "timesheet_202509151540.csv": "client_employee_id",
    "timesheet_202510161121.csv": "client_employee_id",
    "timesheet_202510161122.csv": "client_employee_id",
    "timesheet_202510161124.csv": "client_employee_id"

}

for filename, id_column in TIMESHEET_ID_COLUMNS.items():
    timesheet_df = dataframes[filename]

    timesheet_ids = set(
        timesheet_df[id_column]
        .dropna()
        .astype("string")
        .str.strip()
    )

    timesheet_ids.discard("")

    unmatched = timesheet_ids - roster_ids

    print(f"\n{filename}")
    print("Unique timesheet IDs:", len(timesheet_ids))
    print("Unmatched IDs:", len(unmatched))
    print("Unmatched samples:", list(unmatched)[:20])


timesheet_202509151540.csv
Unique timesheet IDs: 9792
Unmatched IDs: 9767
Unmatched samples: ['500750', '504990', '414644', '104697', '418895', '419854', '507641', '508379', '410340', '503297', '101626', '100987', '502909', '416754', '104538-104538', '503503', '417883', '102686', '504903', '12308']

timesheet_202510161121.csv
Unique timesheet IDs: 8303
Unmatched IDs: 8276
Unmatched samples: ['500750', '504990', '414644', '104697', '418895', '419854', '503297', '100987', '502909', '416754', '503503', '102686', '504903', '12308', '501024', '507204', '12907', '404607', '414083', '421184']

timesheet_202510161122.csv
Unique timesheet IDs: 8270
Unmatched IDs: 8243
Unmatched samples: ['504990', '414644', '104697', '418895', '507641', '419854', '503297', '100987', '502909', '416754', '503503', '102686', '504903', '12308', '501024', '507204', '12907', '404607', '414083', '421184']

timesheet_202510161124.csv
Unique timesheet IDs: 6411
Unmatched IDs: 6394
Unmatched samples: ['504990', '413576'